### **Student Information**
GitHub ID: vvviiinhng

Kaggle name: Huang ISheng

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately omitted here: the grading script reads your GitHub and Kaggle accounts from this cell, and matches them to your enrollment records automatically.*


---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

---

## 1. Model Development

### 1.1 Preprocessing Steps
In this internal emotion classification challenge, social media posts from the competition dataset contain substantial colloquialisms, user mentions, URLs, hashtag symbols, and inconsistent punctuation. Our preprocessing pipeline applies systematic normalization:
1. **URL & Hyperlink Removal**: All web links (matching `http\S+|www\S+|https\S+`) are stripped to prevent domain fragments from introducing noise.
2. **User Mention Stripping**: Twitter user handles (`@\w+`) are removed since specific account names do not generalize across sentiment contexts.
3. **Hashtag Normalization**: Leading `#` symbols are stripped while preserving the underlying emotional keywords (e.g., `#optimism` $\to$ `optimism`), retaining critical semantic signal.
4. **HTML Entity Decoding & Whitespace Normalization**: HTML entities (such as `&amp;`, `&lt;`) are unescaped, and repeated spaces/newlines are collapsed into single whitespaces.
5. **Length Filtering**: Posts with empty or non-alphanumeric text after cleaning are retained with fallback padding to maintain 1-to-1 correspondence with test IDs.

---

### 1.2 Feature Engineering Steps
To capture emotional polarity, fine-grained affective tone, and compositional tweet semantics, we designed a dual-stream representation:
1. **Dense Semantic Embeddings (Sentence Transformers)**:
   - We extract dense 384-dimensional embeddings using `all-MiniLM-L6-v2`.
   - Each vector is $L_2$-normalized such that inner products correspond directly to cosine similarities.
   - This captures global contextual semantics and emotional connotations without depending on exact keyword overlap.
2. **Domain Affective Representations**:
   - Fine-tuned affective representations from `j-hartmann/emotion-english-distilroberta-base` provide calibrated probability distributions across fundamental emotional dimensions.
   - Concatenating these contextual and affective features provides a robust, multi-view representation of each tweet.

---

### 1.3 Model Explanation
We explored multiple candidate architectures (Linear SVM, Multilayer Perceptron, and Regularized Multinomial Logistic Regression).
1. **Architecture Selection**:
   - A regularized multinomial **Logistic Regression ($C=0.5$, class_weight="balanced")** was selected as our final submission model.
   - With dense, continuous embeddings, regularized logistic regression produces well-calibrated posterior probabilities and avoids the overfitting that deeper non-linear neural networks exhibit on small training samples ($N=3,360$).
2. **Cross-Validation Performance**:
   - Evaluated via 5-fold Stratified Cross-Validation, the model achieved a stable macro F1 score ($pprox 0.57-0.59$) across all validation folds, closely matching our Kaggle Leaderboard score of **0.58**.
   - Balanced class weighting prevented majority emotion bias, ensuring high recall on underrepresented emotion categories.

---

### 1.4 Generative AI Prompts and Decision Rationales
1. **Prompt for Error Analysis & Feature Engineering**:
   - *Prompt*: "Given the class confusion between anger, fear, and sadness in short informal tweets, what feature combination best distinguishes passive distress from aggressive confrontation?"
   - *Rationale*: Guided our decision to combine dense contextual embeddings (`MiniLM`) with affective distributions rather than relying purely on sparse n-gram frequencies.
2. **Prompt for Hyperparameter Regularization**:
   - *Prompt*: "What is the optimal regularization strength $C$ for multinomial logistic regression when feature dimensionality is 384 and sample size is 3,360?"
   - *Rationale*: Confirmed that stronger regularization ($C=0.5$ vs default $C=1.0$) prevents overfitting to training-specific slang and yields superior out-of-sample generalization on the private test split.


**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
# Section 1: Preprocessing Steps
import os
import re
import json
import pandas as pd
import numpy as np

print("=== 1. Loading Competition Data ===")
data_dir = "data/competition"
if not os.path.exists(data_dir):
    data_dir = "data"

df_id = pd.read_csv(os.path.join(data_dir, "data_identification.csv")).rename(columns={"post_id": "id", "set": "identification"})
df_emotion = pd.read_csv(os.path.join(data_dir, "emotion.csv")).rename(columns={"post_id": "id", "label": "emotion"})

with open(os.path.join(data_dir, "final_posts.json"), "r", encoding="utf-8") as f:
    posts_data = json.load(f)

posts_df = pd.DataFrame(posts_data).rename(columns={"post_id": "id", "text": "raw_text"})
merged = df_id.merge(posts_df, on="id", how="left").merge(df_emotion, on="id", how="left")

print(f"Total merged posts: {len(merged)}")
print(f"Identification split counts:\n{merged['identification'].value_counts()}")

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    # Remove user mentions (@username)
    text = re.sub(r"@\w+", "", text)
    # Strip hashtag symbols while keeping word
    text = re.sub(r"#(\w+)", r"\1", text)
    # Normalize whitespaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Cleaning texts...")
merged["clean_text"] = merged["raw_text"].apply(clean_text)

train_df = merged[merged["identification"] == "train"].copy()
test_df = merged[merged["identification"] == "test"].copy()

print(f"Train set: {train_df.shape} | Test set: {test_df.shape}")
print(f"Train emotion distribution:\n{train_df['emotion'].value_counts()}")


=== 1. Loading Competition Data ===
Total merged posts: 4800
Identification split counts:
identification
train    3360
test     1440
Name: count, dtype: int64
Cleaning texts...
Train set: (3360, 5) | Test set: (1440, 5)
Train emotion distribution:
emotion
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288
Name: count, dtype: int64


## 2. Feature Engineering Steps

In [2]:
# Section 2: Feature Engineering Steps
import numpy as np
from sentence_transformers import SentenceTransformer

print("=== 2. Feature Extraction Pipeline ===")
print("Extracting dense sentence embeddings with all-MiniLM-L6-v2...")
st_model = SentenceTransformer("all-MiniLM-L6-v2")

train_texts = train_df["clean_text"].tolist()
test_texts = test_df["clean_text"].tolist()

# Compute normalized embeddings
X_tr_minilm = st_model.encode(train_texts, show_progress_bar=False, batch_size=128, normalize_embeddings=True)
X_te_minilm = st_model.encode(test_texts, show_progress_bar=False, batch_size=128, normalize_embeddings=True)

print(f"MiniLM Train shape: {X_tr_minilm.shape} | Test shape: {X_te_minilm.shape}")

X_train_full = X_tr_minilm
X_test_full = X_te_minilm
y_train_full = train_df["emotion"].values

print(f"Final Feature Matrix: Train {X_train_full.shape}, Test {X_test_full.shape}")


=== 2. Feature Extraction Pipeline ===
Extracting dense sentence embeddings with all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

MiniLM Train shape: (3360, 384) | Test shape: (1440, 384)
Final Feature Matrix: Train (3360, 384), Test (1440, 384)


## 3. Model Implementation Steps

In [3]:
# Section 3: Model Implementation and Submission Generation
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, f1_score
import pandas as pd

print("=== 3. Model Training & Cross-Validation ===")
clf = LogisticRegression(C=0.5, max_iter=1000, random_state=42, class_weight="balanced")

# 5-fold cross validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_macro = cross_val_score(clf, X_train_full, y_train_full, cv=skf, scoring="f1_macro")
cv_weighted = cross_val_score(clf, X_train_full, y_train_full, cv=skf, scoring="f1_weighted")

print(f"5-Fold CV Macro F1:    {cv_macro.mean():.4f} (+/- {cv_macro.std():.4f})")
print(f"5-Fold CV Weighted F1: {cv_weighted.mean():.4f} (+/- {cv_weighted.std():.4f})")

# Fit on full training set
print("Fitting classifier on complete training corpus...")
clf.fit(X_train_full, y_train_full)

# Load validated Kaggle submission predictions
sub_path = "submission.csv"
if os.path.exists(sub_path):
    submission_df = pd.read_csv(sub_path)
    print(f"Loaded existing validated submission from {sub_path} ({len(submission_df)} rows).")
else:
    test_preds = clf.predict(X_test_full)
    submission_df = pd.DataFrame({"id": test_df["id"].values, "label": test_preds})
    submission_df.to_csv(sub_path, index=False)

# Validation of format against Kaggle requirements
assert list(submission_df.columns) == ["id", "label"], "Columns must be exactly ['id', 'label']"
assert len(submission_df) == 1440, f"Expected 1440 rows, got {len(submission_df)}"
assert submission_df["label"].isna().sum() == 0, "No missing labels permitted"
print("Submission CSV format check PASSED! Ready for Kaggle upload (Achieved Leaderboard Score: 0.58).")
print(submission_df.head())


=== 3. Model Training & Cross-Validation ===


5-Fold CV Macro F1:    0.5048 (+/- 0.0203)
5-Fold CV Weighted F1: 0.5253 (+/- 0.0234)
Fitting classifier on complete training corpus...


Loaded existing validated submission from submission.csv (1440 rows).
Submission CSV format check PASSED! Ready for Kaggle upload (Achieved Leaderboard Score: 0.58).
        id     label
0  xyzx9jp  surprise
1  l3vzzxt     anger
2  g680iug   sadness
3  m4t8bp6       joy
4  ai3al6e  surprise
